# Fundamentação teórica do capítulo fotovoltaico

<details>
<summary>Descrição e método</summary>

SPGER0046 UFPB. Levantamento de livros, artigos, relatórios e fontes de método para as quatro seções e 19 subseções. Adaptação do notebook multibase WCNPS da pasta Artigo/Bibliometria. Esta versão não busca lacuna científica nem produz uma revisão sistemática automática. As saídas identificam DOI, autoria, ano, periódico ou editora, acesso, tópico e citações separadas por base e data. Livros e artigos não são classificados como se tivessem cobertura bibliométrica idêntica.

</details>

## 0 Preparação e local de trabalho

<details>
<summary>Descrição e método</summary>

O código pode executar em Jupyter ou Colab. Credenciais são lidas de variáveis de ambiente ou dos Secrets do Colab: OPENALEX_API_KEY, SCOPUS_API_KEY, WOS_API_KEY e IEEE_API_KEY. Nenhuma chave é exibida ou gravada. Sem as chaves institucionais, o levantamento continua com Crossref e OpenAlex e registra as bases não executadas. O notebook executado já permite consultar os resultados. NOVA_COLETA=False reutiliza o retrato documentado; True cria outra pasta de respostas e consulta novamente.

</details>

In [ ]:
from pathlib import Path
import sys,subprocess,importlib.util,importlib.metadata,json,io,zipfile,base64,hashlib
from datetime import datetime,timezone,date
requirements={"requests":"requests>=2.32,<3","pandas":"pandas>=2.2,<3","docx":"python-docx>=1.1,<2"}
missing=[requirement for module,requirement in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
import pandas as pd
from IPython.display import display,Markdown,HTML
display(HTML("<style>.text_cell_render,.jp-RenderedHTMLCommon{font:20px/1.65 Arial,sans-serif}summary{cursor:pointer}.dataframe{font-size:17px}td,th{padding:8px}</style>"))
BIB=next((p/"bibliografia" for p in [Path.cwd(),*Path.cwd().parents] if (p/"bibliografia/temas.json").exists()),None)
if BIB is None:BIB=Path.cwd()/"Projeto_FV_PVDAQ_2107_2024/bibliografia"
BIB.mkdir(parents=True,exist_ok=True)
support=base64.b64decode("...")
assert hashlib.sha256(support).hexdigest()=="b7fc7013bf805290bb583d133a180f9029a292ca7ffa69053edfbce24db5d040"
with zipfile.ZipFile(io.BytesIO(support)) as archive:
    for item in archive.infolist():
        target=(BIB/item.filename).resolve();assert target.is_relative_to(BIB.resolve())
        if not target.exists():target.write_bytes(archive.read(item.filename))
sys.path.insert(0,str(BIB))
import bibliografia_fv as bibliography
import importlib;importlib.reload(bibliography)
CONSULTAR_REDE=True
NOVA_COLETA=False
SURVEY=bibliography.Survey(BIB,live=CONSULTAR_REDE)
if NOVA_COLETA:
    SURVEY.raw=BIB/"raw"/("consulta_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
    SURVEY.raw.mkdir(parents=True,exist_ok=True)
    SURVEY.cfg["through_date"]=date.today().isoformat()
print("Saídas:",SURVEY.out)
print("Chaves presentes:",{provider:bool(value) for provider,value in SURVEY.keys.items()})


## 1 Protocolo e consultas dos 19 tópicos

<details>
<summary>Descrição e método</summary>

As consultas do protocolo preservam os tópicos do plano. Obras de método e livros entram como sementes explícitas. A seleção automática exige termos específicos no título, além da pertinência temática. Isso evita promover artigos de células ou física solar a referências sobre inversores ou variabilidade do recurso terrestre somente por terem muitas citações.

</details>

In [ ]:
display(pd.DataFrame(SURVEY.topics)[["id","title","owner","query"]].rename(columns={"id":"Subseção","title":"Tópico","owner":"Responsável","query":"Consulta"}))
print(SURVEY.cfg["ranking"])


## 2 Coleta multibase com rastreabilidade

<details>
<summary>Descrição e método</summary>

As respostas originais ficam em raw e as consultas efetivas em outputs/logs. O retrato padrão é limitado ao protocolo de 16/10/2026. Consultas são limitadas por tópico, sem alegação de exaustividade. A chave pode não conferir acesso ao conteúdo integral ou a todos os campos de uma base. Falhas e ausência de credenciais não são representadas como buscas bem-sucedidas ou zero publicações.

</details>

In [ ]:
REGISTROS=SURVEY.collect()
display(SURVEY.availability)
display(pd.DataFrame(REGISTROS).groupby("source").size().rename("Registros"))
print("Registros normalizados:",len(REGISTROS))


## 3 Deduplicação classificação e seleção de leituras

<details>
<summary>Descrição e método</summary>

DOI normalizado é a primeira chave. Sem DOI, usa-se título normalizado e ano, preservando edições distintas. As citações de bases diferentes não são somadas. A guia combina livro, métodos específicos, trabalhos citados no conjunto recuperado e complemento recente. Leitura integral e conferência de páginas ou equações continuam necessárias antes da redação.

</details>

In [ ]:
OBRAS=SURVEY.consolidate()
LEITURAS=SURVEY.select()
RESUMO=SURVEY.export()
display(pd.DataFrame([RESUMO]).drop(columns=["source_status"]))
display(pd.DataFrame(LEITURAS).groupby("topic").size().rename("Leituras indicadas"))


# 1 Recurso solar

<details>
<summary>Descrição e método</summary>

Referências para a seção 1. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 1.1 Geometria solar e irradiância extraterrestre

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Calcular a posição solar com SPA e a irradiância extraterrestre com Spencer. O azimute é contado a partir do norte no sentido horário.

Limite do estudo de caso: Não há observação independente de ângulo solar; a geometria pode ser calculada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.2 Grandezas radiométricas e componentes da radiação

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Irradiância é potência por área; irradiação é sua integral no tempo. GHI é horizontal, DNI é normal aos raios solares e DHI é difusa horizontal. POA se refere ao plano do arranjo.

Limite do estudo de caso: Sem GHI/DNI/DHI locais medidos; reconstrução não equivale a observação.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.3 Modulação atmosférica e variabilidade espaço temporal

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Nuvens, aerossóis e vapor d’água modificam o recurso solar. A série permite descrever a variabilidade intradiária e entre dias; a seleção dos dias ilustrativos usa apenas POA e cobertura.

Limite do estudo de caso: Não se atribuem causas atmosféricas isoladas nem climatologia anual a um trimestre.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.4 Medição e estimativa do recurso solar

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Inspecionar integridade, unidades, lacunas, simultaneidade e limites físicos. Medições de superfície, satélites, reanálises e atlas têm representatividades diferentes; os produtos em grade são contextualizados, sem uso na cadeia principal.

Limite do estudo de caso: Disponibilidade elevada não certifica sensores. Altura do anemômetro e alguns metadados permanecem incompletos.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 2 Ambiente e montagem

<details>
<summary>Descrição e método</summary>

Referências para a seção 2. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 2.1 Condições térmicas

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. Temperatura ambiente, vento e trocas de calor condicionam a operação do módulo. As distribuições observadas e o cenário de menor ventilação ilustram a passagem do ambiente à modelagem térmica da seção 3.2.

Limite do estudo de caso: U₀ e U₁ genéricos; T operacional é aproximação de célula, sem validação térmica independente.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.2 Albedo e ambiente radiativo da superfície

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. O albedo expressa refletância do entorno. No exercício de transposição, varia-se 0,10/0,20/0,40 mantendo as componentes reconstruídas na referência 0,20. O arranjo real é monofacial; bifacialidade exige caracterização traseira adicional.

Limite do estudo de caso: Expressão isotrópica para reflexão do solo. Não há albedo medido; a leitura POA principal não recebe uma segunda contribuição refletida.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.3 Sombreamento sujidade e obstruções locais

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. A sujeira pode atenuar a luz incidente e diferir entre módulos e sensor POA. Os cenários são hipóteses de atenuação adicional à leitura. Sombreamento parcial pode envolver mismatch e diodos de bypass, que um redutor uniforme não reproduz.

Limite do estudo de caso: Sem histórico de limpeza e mapa de sombras; o resíduo não identifica a perda real de sujidade ou sombra.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.4 Geometria de instalação e exposição do arranjo fotovoltaico

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. A instalação define exposição e incidência. A inclinação é medida em relação à horizontal. Espaçamento entre fileiras e exposição traseira precisam de geometria adicional; em um exemplo de sombra coplanar, uma altura h gera projeção horizontal h/tan(altura solar).

Limite do estudo de caso: A expressão de projeção é um exemplo plano sem dimensão de fileira real; não calcula o sombreamento específico da usina.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 3 Conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Referências para a seção 3. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 3.1 Transposição para o plano do arranjo

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A cadeia usual transpõe GHI/DNI/DHI para POA. Como a instalação já mede POA, o modelo principal começa dessa observação. A inversão opcional usa Driesse 2024, Erbs–Driesse e Perez–Driesse para exercícios de orientação/albedo.

Limite do estudo de caso: Há possibilidade de soluções não únicas; o fechamento de POA é teste numérico, sem validação independente das componentes.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.2 Modelagem da temperatura operacional

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A temperatura estimada entra no modelo elétrico. U₀=25 W/(m² K) e U₁=6,84 W s/(m³ K) são parâmetros genéricos de referência; a sensibilidade de ventilação usa 15 e 3, respectivamente.

Limite do estudo de caso: Ausência de temperatura de módulo/célula e altura de vento suficientemente documentada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.3 Modelagem elétrica do módulo fotovoltaico

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. O modelo de diodo único fornece curvas I–V, P–V e o ponto de máxima potência. Compare irradiâncias a 25 °C e temperaturas a 1.000 W/m². Registre divergências de classificação e parâmetros entre CEC e metadados.

Limite do estudo de caso: Sem varreduras I–V experimentais da instalação; as curvas não constituem validação do módulo.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.4 Modelagem do arranjo fotovoltaico em corrente contínua

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. Em uma associação ideal, as tensões somam em série e as correntes em paralelo. O exemplo 20 módulos × 6 strings por grupo corresponde a 120 módulos, mas não confirma o cabeamento real ou a ligação aos MPPTs.

Limite do estudo de caso: Strings/MPPTs reais incompletos; V×I disponível não foi adotado como potência DC observada certificada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.5 Conversão de CC para CA e modelagem do inversor

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A curva PVWatts estima conversão e clipping. O limite de referência é 30 kW por inversor; 27,6 kW permanece como sensibilidade documentada. Verificar a faixa real de MPPT exige topologia e configuração de campo adicionais.

Limite do estudo de caso: A simulação agregada não verifica todas as restrições reais de tensão/MPPT nem confirma o limite instalado.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.5"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.6 Integração temporal e estimativa de energia

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. Integre potência média por duração de cada intervalo. Compare séries e energias nas mesmas janelas. Apresente energia mensal e trimestral observada completa separadamente dos intervalos diurnos pareados.

Limite do estudo de caso: O recorte é trimestral, não uma medição anual; energia pareada não equivale ao total integral do dia.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.6"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 4 Desempenho e validação

<details>
<summary>Descrição e método</summary>

Referências para a seção 4. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 4.1 Balanço energético e cadeia de perdas

<details>
<summary>Descrição e método</summary>

Responsável: Júlia Ribeiro. Integre a cadeia óptica, térmica, elétrica DC, conversão e limite AC. A diferença entre estágios pertence ao modelo; um desvio do medidor pode envolver fenômenos e incertezas não identificados.

Limite do estudo de caso: Não se identificam perdas ópticas/operacionais reais isoladas sem medições adicionais; ganhos sob frio podem ocorrer.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.2 Produtividades de referência do arranjo e final

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Yr é o recurso em horas equivalentes; Ya representa a energia DC por capacidade; Yf representa energia AC por capacidade. HSP/PSH corresponde ao mesmo conceito de recurso quando plano e janela são os mesmos.

Limite do estudo de caso: Ya é modelada; HSP da janela pareada não é duração de brilho solar nem total integral diário.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.3 Razão de desempenho e decomposição normalizada das perdas

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Compare PR observado e modelado com irradiação e energia da mesma janela. Lc e Ls decompõem o modelo em perdas de captura e sistema. Informe a normalização e o papel da potência declarada.

Limite do estudo de caso: Não combinar energia integral do trimestre com POA parcial; perdas calculadas não são identificação experimental de cada causa.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.4 Validação de modelos de geração fotovoltaica

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Avalie viés, magnitude do erro, correlação e determinação. Erro=simulado−observado. MBE, MAE e RMSE são em kW; nMBE/nRMSE são divididos pela média observada dos respectivos pares. O modelo físico é separado do fator empírico ajustado somente em abril.

Limite do estudo de caso: Junho é teste temporal separado, não instalação independente. Correlação elevada não prova exatidão; nenhum limiar universal de aceitação foi adotado.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.5 Avaliação de desempenho de sistemas integrados à edificação

<details>
<summary>Descrição e método</summary>

Responsável: Júlia Ribeiro. Explore fachadas sul/leste/oeste e menor ventilação. O exemplo separado de 100 m² semitransparentes usa transmissão solar 0,25, reflexão 0,12, absorção 0,63 e transmissão visível 0,35. As propriedades não caracterizam o módulo real.

Limite do estudo de caso: Somente horas elegíveis da inversão; sem fachada medida, SHGC certificado, cálculo de lux ou balanço térmico completo do edifício.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.5"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 5 Documentos exportações e execução

<details>
<summary>Descrição e método</summary>

Word por subseção; Markdown e CSV por seção; catálogo consolidado e relações por tópico; BibTeX e RIS para gerenciadores de referências. Os arquivos BibTeX e RIS permitem importar as referências em gerenciadores bibliográficos. Os índices por seção orientam a consulta.

</details>

In [ ]:
print("Word:",SURVEY.out/"Referencias_por_Subsecao.docx")
print("BibTeX:",SURVEY.out/"referencias_fundamentacao.bib")
print("RIS:",SURVEY.out/"referencias_fundamentacao.ris")
display(pd.DataFrame([{"Arquivo":str(p.relative_to(SURVEY.out)),"Bytes":p.stat().st_size} for p in sorted(SURVEY.out.rglob("*")) if p.is_file()]))
assert set(w["topic"] for w in SURVEY.selected)==set(t["id"] for t in SURVEY.topics)
assert all(w["title"] and (w["doi"] or w["url"]) for w in SURVEY.selected)
assert not any("Solar Dynamics Observatory" in w["title"] or "polymer tandem" in w["title"].lower() for w in SURVEY.selected)
print("Conferência: 19 tópicos com referências efetivas e origem identificada.")
